# Energinet (Denmark) – Complex Bid Groups

This notebook demonstrates the two complex bid group types supported by Energinet:

1. **Exclusive group** (`exclusiveBidsIdentification`) — a set of bids where
   at most one can be activated.  Useful for alternative resources that cannot
   run simultaneously (e.g. two wind farms competing for the same activation slot).

2. **Multipart bid** (`multipartBidIdentification`) — multiple price tiers for
   the same MTU on the same resource.  If the highest-priced tier is activated,
   all lower-priced tiers must also be activated.  Useful for resources with
   steeply rising marginal costs.

Both group types share a common builder API:
```
GroupClass(bidding_zone=...)
    .direction(...)
    .resource(...)
    .product_type(...)
    .for_mtu(...)
    .add_component(...)
    .add_component(...)
    .build()  → tuple[BidTimeSeriesModel, ...]
```

## Energinet specifics for complex bids

- `mktPSRType.psrType` is **mandatory** on every bid time series (pass `psr_type=ProductionType.X` per component)
- Inclusive groups are **not supported** by Energinet
- Conditional link types A71 and A72 are not supported

## Imports

In [1]:
from nexa_mfrr_eam import (
    BidDocument,
    BiddingZone,
    Direction,
    ExclusiveGroup,
    MARIMode,
    MarketProductType,
    MultipartGroup,
    ProductionType,
    TSO,
)

# Synthetic Energinet BSP/BRP party ID
BSP_PARTY_ID = "10XBRP-001-----A"
BSP_CODING_SCHEME = "A01"

# Two competing wind resources in DK1
WIND_A = "10XWINDA-DK1----W"
WIND_B = "10XWINDB-DK1----W"

## Part 1: Exclusive Group

An exclusive group lets you submit bids from two (or more) resources that
cannot both be activated in the same quarter.  The TSO will activate at most
one bid in the group per MTU.  Typical use cases:

- Two wind farms sharing a congested grid connection
- A battery and a backup generator with a shared transformer

### Constraints
- All components must cover the **same MTU**
- All components must have the **same product type**
- All components must belong to the **same bidding zone**
- At least **2 components** required

Prices can differ between components (each resource has its own merit order
price).

### Example: two competing wind farms in DK1

In [2]:
exclusive_group = (
    ExclusiveGroup(bidding_zone=BiddingZone.DK1)
    .direction(Direction.UP)
    .product_type(MarketProductType.SCHEDULED_ONLY)
    .for_mtu("2026-03-21T10:00Z")
    .add_component(
        volume_mw=40,
        price_eur=62.50,
        divisible=True,
        min_volume_mw=10,
        resource_id=WIND_A,
        resource_coding_scheme="A01",
        psr_type=ProductionType.WIND_ONSHORE,  # B19
    )
    .add_component(
        volume_mw=60,
        price_eur=71.00,
        divisible=True,
        min_volume_mw=15,
        resource_id=WIND_B,
        resource_coding_scheme="A01",
        psr_type=ProductionType.WIND_OFFSHORE,  # B18
    )
    .build()
)

print(f"Group ID: {exclusive_group[0].exclusive_bids_identification}")
print(f"Components: {len(exclusive_group)}")
for i, bid in enumerate(exclusive_group, 1):
    print(f"  Component {i}: {bid.period.point.quantity} MW "
          f"@ {bid.period.point.energy_price} EUR/MWh  "
          f"psr_type={bid.psr_type}  "
          f"resource={bid.registered_resource_mrid}")

Group ID: ee2bb17e-e11e-40ab-8811-d769b50c3ae3
Components: 2
  Component 1: 40 MW @ 62.5 EUR/MWh  psr_type=B19  resource=10XWINDA-DK1----W
  Component 2: 60 MW @ 71.0 EUR/MWh  psr_type=B18  resource=10XWINDB-DK1----W


In [3]:
doc_exclusive = (
    BidDocument(tso=TSO.ENERGINET)
    .sender(party_id=BSP_PARTY_ID, coding_scheme=BSP_CODING_SCHEME)
    .add_bids(exclusive_group)
    .build()
)

errors = doc_exclusive.validate(mari_mode=MARIMode.PRE_MARI)
if errors:
    for e in errors:
        print(f"ERROR: {e}")
else:
    print(f"Document valid: {doc_exclusive.time_series_count} bids")

Document valid: 2 bids


### Preview XML for exclusive group

Notice `<exclusiveBidsIdentification>` and `<mktPSRType.psrType>` in each
`Bid_TimeSeries`.

In [4]:
xml_excl = doc_exclusive.to_xml().decode("utf-8")
print(xml_excl[:3000])

<?xml version='1.0' encoding='UTF-8'?>
<ReserveBid_MarketDocument xmlns="urn:iec62325.351:tc57wg16:451-7:reservebiddocument:7:4">
  <mRID>59092d9f-faa4-470b-9810-21c31b94a914</mRID>
  <revisionNumber>1</revisionNumber>
  <type>A37</type>
  <process.processType>A47</process.processType>
  <sender_MarketParticipant.mRID codingScheme="A01">10XBRP-001-----A</sender_MarketParticipant.mRID>
  <sender_MarketParticipant.marketRole.type>A46</sender_MarketParticipant.marketRole.type>
  <receiver_MarketParticipant.mRID codingScheme="A01">10X1001A1001A248</receiver_MarketParticipant.mRID>
  <receiver_MarketParticipant.marketRole.type>A34</receiver_MarketParticipant.marketRole.type>
  <createdDateTime>2026-03-25T09:40:56Z</createdDateTime>
  <reserveBid_Period.timeInterval>
    <start>2026-03-21T10:00Z</start>
    <end>2026-03-21T10:15Z</end>
  </reserveBid_Period.timeInterval>
  <domain.mRID codingScheme="A01">10Y1001A1001A796</domain.mRID>
  <subject_MarketParticipant.mRID codingScheme="A01">10XB

## Part 2: Multipart Bid

A multipart bid groups price tiers for the same resource and MTU.  All tiers
share a `multipartBidIdentification`.  If a higher-priced tier is selected by
the AOF, all cheaper tiers are automatically selected too.

### Constraints
- All components: **same direction**, **same MTU**, **same product type**, **same bidding zone**
- All component prices must be **distinct**
- At least **2 components** required

### Example: offshore wind farm with three price tiers

The first 20 MW are cheap to activate (marginal cost close to zero).  The next
15 MW require more curtailment.  The final 10 MW are the most expensive tier.

In [5]:
multipart_group = (
    MultipartGroup(bidding_zone=BiddingZone.DK1)
    .direction(Direction.UP)
    .resource(WIND_B, coding_scheme="A01")
    .product_type(MarketProductType.SCHEDULED_ONLY)
    .for_mtu("2026-03-21T10:00Z")
    .add_component(
        volume_mw=20,
        price_eur=45.00,   # cheapest tier
        divisible=True,
        min_volume_mw=5,
        psr_type=ProductionType.WIND_OFFSHORE,
    )
    .add_component(
        volume_mw=15,
        price_eur=70.00,   # mid tier
        divisible=True,
        min_volume_mw=5,
        psr_type=ProductionType.WIND_OFFSHORE,
    )
    .add_component(
        volume_mw=10,
        price_eur=110.00,  # expensive tier
        divisible=False,
        psr_type=ProductionType.WIND_OFFSHORE,
    )
    .build()
)

print(f"Group ID: {multipart_group[0].multipart_bid_identification}")
print(f"Tiers:    {len(multipart_group)}")
for i, bid in enumerate(multipart_group, 1):
    print(f"  Tier {i}: {bid.period.point.quantity} MW "
          f"@ {bid.period.point.energy_price} EUR/MWh  "
          f"({'divisible' if bid.divisible_code == 'A01' else 'indivisible'})")

Group ID: 84296be2-7e68-4d24-8965-2f82b9e817c9
Tiers:    3
  Tier 1: 20 MW @ 45.0 EUR/MWh  (divisible)
  Tier 2: 15 MW @ 70.0 EUR/MWh  (divisible)
  Tier 3: 10 MW @ 110.0 EUR/MWh  (indivisible)


In [6]:
doc_multipart = (
    BidDocument(tso=TSO.ENERGINET)
    .sender(party_id=BSP_PARTY_ID, coding_scheme=BSP_CODING_SCHEME)
    .add_bids(multipart_group)
    .build()
)

errors = doc_multipart.validate(mari_mode=MARIMode.PRE_MARI)
if errors:
    for e in errors:
        print(f"ERROR: {e}")
else:
    print(f"Document valid: {doc_multipart.time_series_count} bids")

xml_mp = doc_multipart.to_xml().decode("utf-8")
print(xml_mp[:2000])

Document valid: 3 bids
<?xml version='1.0' encoding='UTF-8'?>
<ReserveBid_MarketDocument xmlns="urn:iec62325.351:tc57wg16:451-7:reservebiddocument:7:4">
  <mRID>67f7ab10-6c65-4c31-beaf-955ee3d4e75b</mRID>
  <revisionNumber>1</revisionNumber>
  <type>A37</type>
  <process.processType>A47</process.processType>
  <sender_MarketParticipant.mRID codingScheme="A01">10XBRP-001-----A</sender_MarketParticipant.mRID>
  <sender_MarketParticipant.marketRole.type>A46</sender_MarketParticipant.marketRole.type>
  <receiver_MarketParticipant.mRID codingScheme="A01">10X1001A1001A248</receiver_MarketParticipant.mRID>
  <receiver_MarketParticipant.marketRole.type>A34</receiver_MarketParticipant.marketRole.type>
  <createdDateTime>2026-03-25T09:40:56Z</createdDateTime>
  <reserveBid_Period.timeInterval>
    <start>2026-03-21T10:00Z</start>
    <end>2026-03-21T10:15Z</end>
  </reserveBid_Period.timeInterval>
  <domain.mRID codingScheme="A01">10Y1001A1001A796</domain.mRID>
  <subject_MarketParticipant.mRID 

## Part 3: Combined Document – Exclusive Group + Multipart Bid

A real bid submission may include both group types in a single document.
Here we combine the exclusive group from Part 1 and the multipart bid from
Part 2 — they are for different resources so there is no conflict.

In [7]:
doc_combined = (
    BidDocument(tso=TSO.ENERGINET)
    .sender(party_id=BSP_PARTY_ID, coding_scheme=BSP_CODING_SCHEME)
    .add_bids(exclusive_group)   # 2 bids: exclusiveBidsIdentification
    .add_bids(multipart_group)   # 3 bids: multipartBidIdentification
    .build()
)

errors = doc_combined.validate(mari_mode=MARIMode.PRE_MARI)
print(f"Total bids: {doc_combined.time_series_count}")
print(f"Validation errors: {len(errors)}")
if errors:
    for e in errors:
        print(f"  {e}")

Total bids: 5
Validation errors: 0


## Part 4: Validation – Constraint Violations

The builders enforce constraints at `build()` time and raise
`BidValidationError` with a list of violations.

In [8]:
from nexa_mfrr_eam.exceptions import BidValidationError

# Multipart: duplicate prices are not allowed
try:
    (
        MultipartGroup(bidding_zone=BiddingZone.DK1)
        .direction(Direction.UP)
        .resource(WIND_B)
        .product_type(MarketProductType.SCHEDULED_ONLY)
        .for_mtu("2026-03-21T10:00Z")
        .add_component(volume_mw=20, price_eur=50.0, divisible=False,
                       psr_type=ProductionType.WIND_OFFSHORE)
        .add_component(volume_mw=15, price_eur=50.0, divisible=False,  # duplicate!
                       psr_type=ProductionType.WIND_OFFSHORE)
        .build()
    )
except BidValidationError as exc:
    print("MultipartGroup duplicate price:")
    for e in exc.errors:
        print(f"  {e}")

print()

# ExclusiveGroup: only one component (requires at least 2)
try:
    (
        ExclusiveGroup(bidding_zone=BiddingZone.DK1)
        .direction(Direction.UP)
        .product_type(MarketProductType.SCHEDULED_ONLY)
        .for_mtu("2026-03-21T10:00Z")
        .add_component(volume_mw=30, price_eur=60.0, divisible=False,
                       psr_type=ProductionType.WIND_ONSHORE)
        .build()
    )
except BidValidationError as exc:
    print("ExclusiveGroup single component:")
    for e in exc.errors:
        print(f"  {e}")

MultipartGroup duplicate price:
  MultipartGroup: all component prices must be distinct

ExclusiveGroup single component:
  ExclusiveGroup requires at least 2 components


## Summary

| Group type | Builder | XML element | Constraint |
|------------|---------|-------------|------------|
| Exclusive | `ExclusiveGroup(...).add_component(...).build()` | `<exclusiveBidsIdentification>` | Same MTU, product type, zone; at most 1 activated |
| Multipart | `MultipartGroup(...).add_component(...).build()` | `<multipartBidIdentification>` | Same MTU/direction/zone; distinct prices |
| Inclusive | `InclusiveGroup(...)` — **not supported by Energinet** | `<inclusiveBidsIdentification>` | Fingrid / Statnett / SVK only |

**Production type is mandatory for all Energinet bids** — pass
`psr_type=ProductionType.X` to each `add_component()` call.